# 08 Train GraphEdgeClassifier - Focal Loss 

Notebook ini dibuat untuk tujuan research NFT wash trading detection dengan **GraphEdgeClassifier + focal loss**.

Fokus utama notebook ini:
- memakai split baru: `train_1_50.csv`, `val_temporal.csv`, `test_temporal.csv`
- validation dipakai untuk model selection dan threshold tuning
- test hanya dipakai untuk final evaluation
- **menghapus label leakage** dari rule-based columns
- menambahkan temporal/pair features yang tidak memakai label
- focal loss dibuat lebih stabil memakai `sqrt/capped focal_loss`
- evaluasi diarahkan ke high-recall candidate detection, FP budget, top-k ranking, dan lift vs random

In [1]:
import json
import random
import copy
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [2]:
# =========================
# PATH CONFIG
# =========================

DATA_DIR = Path("../data/processed")
OUTPUT_METRICS_DIR = Path("../outputs/metrics")
OUTPUT_MODELS_DIR = Path("../outputs/models")

OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train_1_50.csv"
VAL_PATH   = DATA_DIR / "val_temporal.csv"
TEST_PATH  = DATA_DIR / "test_temporal.csv"

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(p, "exists=", p.exists())
    if not p.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {p}. Jalankan dulu notebook split terbaru.")

..\data\processed\train_1_50.csv exists= True
..\data\processed\val_temporal.csv exists= True
..\data\processed\test_temporal.csv exists= True


In [3]:
# =========================
# LOAD SPLIT DATA
# =========================

train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
val_df   = pd.read_csv(VAL_PATH, low_memory=False)
test_df  = pd.read_csv(TEST_PATH, low_memory=False)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("Columns:")
print(train_df.columns.tolist())

Train: (35802, 25)
Val  : (149079, 25)
Test : (298160, 25)
Columns:
['transaction_hash', 'block_number', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions', 'timestamp_dt', 'mint_timestamp_dt', 'month', 'is_wash_trading', 'rule_self_trade', 'rule_seller_buyback', 'rule_multi_hop_cycle', 'rule_high_pair_count', 'wash_score', 'confidence_category', 'label_final']


In [4]:
# =========================
# COLUMN CONFIG
# =========================

SOURCE_COL = "from_address"
TARGET_COL = "to_address"
LABEL_COL = "label_final"
TIMESTAMP_COL = "timestamp"
VALUE_COL = "transaction_value" if "transaction_value" in train_df.columns else None
TOKEN_COL = "token_id" if "token_id" in train_df.columns else None
NFT_COL = "nft_address" if "nft_address" in train_df.columns else None
MINT_TIMESTAMP_COL = "mint_timestamp" if "mint_timestamp" in train_df.columns else None

required_cols = [SOURCE_COL, TARGET_COL, LABEL_COL, TIMESTAMP_COL]
missing = [c for c in required_cols if c not in train_df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

for df in [train_df, val_df, test_df]:
    df[TIMESTAMP_COL] = pd.to_numeric(df[TIMESTAMP_COL], errors="coerce")
    df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce").fillna(0).astype(int)
    if VALUE_COL is not None:
        df[VALUE_COL] = pd.to_numeric(df[VALUE_COL], errors="coerce").fillna(0)
    if MINT_TIMESTAMP_COL is not None:
        df[MINT_TIMESTAMP_COL] = pd.to_numeric(df[MINT_TIMESTAMP_COL], errors="coerce")

train_df = train_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
val_df   = val_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)
test_df  = test_df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(name, df[LABEL_COL].value_counts(dropna=False).to_dict(), "fraud_rate=", float(df[LABEL_COL].mean()))
    print("  timestamp:", df[TIMESTAMP_COL].min(), "->", df[TIMESTAMP_COL].max())

Train {0: 35100, 1: 702} fraud_rate= 0.0196078431372549
  timestamp: 1622507092 -> 1629695254
Val {0: 149027, 1: 52} fraud_rate= 0.0003488083499352692
  timestamp: 1629695273 -> 1629978053
Test {0: 298096, 1: 64} fraud_rate= 0.00021464985242822645
  timestamp: 1629978053 -> 1630454395


## Feature engineering tanpa label leakage

Fitur rule-based seperti `wash_score`, `rule_*`, `is_wash_trading`, dan `confidence_category` **tidak dipakai sebagai input model** karena kolom tersebut membentuk `label_final`.

Notebook ini menambahkan fitur temporal/pair berbasis riwayat masa lalu saja, misalnya jumlah transaksi wallet sebelumnya, jumlah interaksi pair sebelumnya, dan waktu sejak transaksi terakhir.

In [5]:
# =========================
# TEMPORAL / PAIR FEATURES - PAST ONLY, NO LABEL USAGE
# =========================

ENGINEERED_FEATURES = [
    "src_tx_count_past",
    "dst_tx_count_past",
    "src_to_dst_count_past",
    "dst_to_src_count_past",
    "pair_undirected_count_past",
    "token_tx_count_past",
    "nft_tx_count_past",
    "time_since_src_last",
    "time_since_dst_last",
    "log_time_since_src_last",
    "log_time_since_dst_last",
    "src_value_sum_past",
    "dst_value_sum_past",
    "pair_value_sum_past",
    "src_value_mean_past",
    "dst_value_mean_past",
    "pair_value_mean_past",
    "mint_age_seconds",
    "log_transaction_value",
]


def init_temporal_state():
    return {
        "wallet_count": defaultdict(int),
        "wallet_last_time": {},
        "wallet_value_sum": defaultdict(float),
        "pair_directed_count": defaultdict(int),
        "pair_undirected_count": defaultdict(int),
        "pair_value_sum": defaultdict(float),
        "token_count": defaultdict(int),
        "nft_count": defaultdict(int),
    }


def add_past_temporal_features(df, state=None, update_state=True):
    """Tambah fitur berbasis event masa lalu.

    Untuk train: state kosong, update per transaksi.
    Untuk val: state hasil train, update per transaksi val.
    Untuk test: state hasil train+val, update per transaksi test.

    Tidak ada label yang digunakan.
    """
    if state is None:
        state = init_temporal_state()
    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True).copy()

    rows = {name: [] for name in ENGINEERED_FEATURES}

    for _, row in df.iterrows():
        src = str(row[SOURCE_COL]) if pd.notna(row[SOURCE_COL]) else "UNKNOWN_WALLET"
        dst = str(row[TARGET_COL]) if pd.notna(row[TARGET_COL]) else "UNKNOWN_WALLET"
        ts = float(row[TIMESTAMP_COL]) if pd.notna(row[TIMESTAMP_COL]) else 0.0
        value = float(row[VALUE_COL]) if VALUE_COL is not None and pd.notna(row[VALUE_COL]) else 0.0
        token = str(row[TOKEN_COL]) if TOKEN_COL is not None and pd.notna(row[TOKEN_COL]) else "UNKNOWN_TOKEN"
        nft = str(row[NFT_COL]) if NFT_COL is not None and pd.notna(row[NFT_COL]) else "UNKNOWN_NFT"
        pair_dir = (src, dst)
        pair_rev = (dst, src)
        pair_undir = tuple(sorted([src, dst]))

        src_count = state["wallet_count"][src]
        dst_count = state["wallet_count"][dst]
        pair_dir_count = state["pair_directed_count"][pair_dir]
        pair_rev_count = state["pair_directed_count"][pair_rev]
        pair_undir_count = state["pair_undirected_count"][pair_undir]
        token_count = state["token_count"][token]
        nft_count = state["nft_count"][nft]

        src_last = state["wallet_last_time"].get(src, np.nan)
        dst_last = state["wallet_last_time"].get(dst, np.nan)
        time_since_src = ts - src_last if pd.notna(src_last) else 0.0
        time_since_dst = ts - dst_last if pd.notna(dst_last) else 0.0
        time_since_src = max(float(time_since_src), 0.0)
        time_since_dst = max(float(time_since_dst), 0.0)

        src_value_sum = state["wallet_value_sum"][src]
        dst_value_sum = state["wallet_value_sum"][dst]
        pair_value_sum = state["pair_value_sum"][pair_undir]

        src_value_mean = src_value_sum / src_count if src_count > 0 else 0.0
        dst_value_mean = dst_value_sum / dst_count if dst_count > 0 else 0.0
        pair_value_mean = pair_value_sum / pair_undir_count if pair_undir_count > 0 else 0.0

        if MINT_TIMESTAMP_COL is not None and pd.notna(row[MINT_TIMESTAMP_COL]):
            mint_age = max(ts - float(row[MINT_TIMESTAMP_COL]), 0.0)
        else:
            mint_age = 0.0

        rows["src_tx_count_past"].append(src_count)
        rows["dst_tx_count_past"].append(dst_count)
        rows["src_to_dst_count_past"].append(pair_dir_count)
        rows["dst_to_src_count_past"].append(pair_rev_count)
        rows["pair_undirected_count_past"].append(pair_undir_count)
        rows["token_tx_count_past"].append(token_count)
        rows["nft_tx_count_past"].append(nft_count)
        rows["time_since_src_last"].append(time_since_src)
        rows["time_since_dst_last"].append(time_since_dst)
        rows["log_time_since_src_last"].append(np.log1p(time_since_src))
        rows["log_time_since_dst_last"].append(np.log1p(time_since_dst))
        rows["src_value_sum_past"].append(src_value_sum)
        rows["dst_value_sum_past"].append(dst_value_sum)
        rows["pair_value_sum_past"].append(pair_value_sum)
        rows["src_value_mean_past"].append(src_value_mean)
        rows["dst_value_mean_past"].append(dst_value_mean)
        rows["pair_value_mean_past"].append(pair_value_mean)
        rows["mint_age_seconds"].append(mint_age)
        rows["log_transaction_value"].append(np.log1p(max(value, 0.0)))

        if update_state:
            state["wallet_count"][src] += 1
            state["wallet_count"][dst] += 1
            state["wallet_last_time"][src] = ts
            state["wallet_last_time"][dst] = ts
            state["wallet_value_sum"][src] += value
            state["wallet_value_sum"][dst] += value
            state["pair_directed_count"][pair_dir] += 1
            state["pair_undirected_count"][pair_undir] += 1
            state["pair_value_sum"][pair_undir] += value
            state["token_count"][token] += 1
            state["nft_count"][nft] += 1

    for name, values in rows.items():
        df[name] = np.asarray(values, dtype=np.float32)

    return df, state

state0 = init_temporal_state()
train_df, state_after_train = add_past_temporal_features(train_df, state0, update_state=True)
val_df, state_after_val = add_past_temporal_features(val_df, copy.deepcopy(state_after_train), update_state=True)
test_df, _ = add_past_temporal_features(test_df, copy.deepcopy(state_after_val), update_state=True)

print("Engineered features added:", len(ENGINEERED_FEATURES))
train_df[ENGINEERED_FEATURES].head()

Engineered features added: 19


,src_tx_count_past,dst_tx_count_past,src_to_dst_count_past,dst_to_src_count_past,pair_undirected_count_past,token_tx_count_past,nft_tx_count_past,time_since_src_last,time_since_dst_last,log_time_since_src_last,log_time_since_dst_last,src_value_sum_past,dst_value_sum_past,pair_value_sum_past,src_value_mean_past,dst_value_mean_past,pair_value_mean_past,mint_age_seconds,log_transaction_value
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.321579
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3287582.0,37.246826
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.450798
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34.563469
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.227657


In [6]:
# =========================
# NODE MAPPING
# =========================
# Mapping dibuat dari train+val+test supaya wallet baru di validation/test tetap punya ID.
# Ini hanya memakai daftar wallet, bukan label atau fitur masa depan untuk training.

all_nodes = pd.concat([
    train_df[SOURCE_COL], train_df[TARGET_COL],
    val_df[SOURCE_COL], val_df[TARGET_COL],
    test_df[SOURCE_COL], test_df[TARGET_COL],
], axis=0).astype(str).fillna("UNKNOWN_WALLET").unique()

node_to_id = {node: idx for idx, node in enumerate(all_nodes)}
num_nodes = len(node_to_id)
print("Num nodes:", num_nodes)


def map_nodes(df):
    src = df[SOURCE_COL].astype(str).fillna("UNKNOWN_WALLET").map(node_to_id)
    dst = df[TARGET_COL].astype(str).fillna("UNKNOWN_WALLET").map(node_to_id)
    if src.isna().any() or dst.isna().any():
        raise ValueError(f"Ada node yang tidak ada di mapping. missing_src={src.isna().sum()}, missing_dst={dst.isna().sum()}")
    return src.astype(np.int64).values, dst.astype(np.int64).values

Num nodes: 126505


In [7]:
# =========================
# FEATURE SELECTION - STRICT NO LEAKAGE
# =========================

leakage_cols = {
    # target / direct label leakage
    LABEL_COL,
    "is_wash_trading",

    # rule-based columns used to create label_final
    "rule_self_trade",
    "rule_seller_buyback",
    "rule_multi_hop_cycle",
    "rule_high_pair_count",
    "wash_score",
    "confidence_category",
}

id_time_cols = {
    SOURCE_COL,
    TARGET_COL,
    TIMESTAMP_COL,
    "transaction_hash",
    "nft_address",
    "token_id",
    "timestamp_dt",
    "mint_timestamp_dt",
}

# Jangan pakai timestamp mentah sebagai feature, tapi boleh pakai engineered time deltas.
exclude_cols = leakage_cols | id_time_cols

numeric_cols = train_df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

# Pastikan engineered features masuk.
for c in ENGINEERED_FEATURES:
    if c not in feature_cols and c in train_df.columns:
        feature_cols.append(c)

# Remove duplicates while preserving order.
feature_cols = list(dict.fromkeys(feature_cols))

leaked = sorted(set(feature_cols) & leakage_cols)
if leaked:
    raise ValueError(f"Masih ada leakage columns di feature_cols: {leaked}")
if len(feature_cols) == 0:
    raise ValueError("Tidak ada numeric feature yang tersedia.")

print("Num feature cols:", len(feature_cols))
print("Features used:")
for c in feature_cols:
    print("-", c)

print("Excluded leakage columns:")
for c in sorted(leakage_cols):
    print("-", c)

Num feature cols: 28
Features used:
- block_number
- transaction_value
- mint_timestamp
- transfers_out_from
- transfers_in_from
- transfers_out_to
- transfers_in_to
- num_transitions
- month
- src_tx_count_past
- dst_tx_count_past
- src_to_dst_count_past
- dst_to_src_count_past
- pair_undirected_count_past
- token_tx_count_past
- nft_tx_count_past
- time_since_src_last
- time_since_dst_last
- log_time_since_src_last
- log_time_since_dst_last
- src_value_sum_past
- dst_value_sum_past
- pair_value_sum_past
- src_value_mean_past
- dst_value_mean_past
- pair_value_mean_past
- mint_age_seconds
- log_transaction_value
Excluded leakage columns:
- confidence_category
- is_wash_trading
- label_final
- rule_high_pair_count
- rule_multi_hop_cycle
- rule_self_trade
- rule_seller_buyback
- wash_score


In [8]:

# =========================
# NO-LEAKAGE SAFETY CHECK
# =========================

forbidden_features = {
    "is_wash_trading",
    "label_final",
    "rule_self_trade",
    "rule_seller_buyback",
    "rule_multi_hop_cycle",
    "rule_high_pair_count",
    "wash_score",
    "confidence_category",
}

leaked_features = [c for c in feature_cols if c in forbidden_features]
assert len(leaked_features) == 0, f"Leakage features masih masuk: {leaked_features}"

print("No-leakage check passed.")
print("Feature columns used:", feature_cols)


No-leakage check passed.
Feature columns used: ['block_number', 'transaction_value', 'mint_timestamp', 'transfers_out_from', 'transfers_in_from', 'transfers_out_to', 'transfers_in_to', 'num_transitions', 'month', 'src_tx_count_past', 'dst_tx_count_past', 'src_to_dst_count_past', 'dst_to_src_count_past', 'pair_undirected_count_past', 'token_tx_count_past', 'nft_tx_count_past', 'time_since_src_last', 'time_since_dst_last', 'log_time_since_src_last', 'log_time_since_dst_last', 'src_value_sum_past', 'dst_value_sum_past', 'pair_value_sum_past', 'src_value_mean_past', 'dst_value_mean_past', 'pair_value_mean_past', 'mint_age_seconds', 'log_transaction_value']


In [9]:
# =========================
# FEATURE SCALING
# =========================
# Fit scaler hanya di train, lalu transform val/test.

def make_feature_matrix(df, feature_cols):
    return (
        df[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .astype(np.float32)
        .values
    )

scaler = StandardScaler()
X_train_raw = make_feature_matrix(train_df, feature_cols)
X_val_raw = make_feature_matrix(val_df, feature_cols)
X_test_raw = make_feature_matrix(test_df, feature_cols)

X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print("Any NaN train:", np.isnan(X_train).any())

X_train: (35802, 28)
X_val  : (149079, 28)
X_test : (298160, 28)
Any NaN train: False


In [10]:
# =========================
# DATASET & DATALOADER
# =========================

class EdgeDataset(Dataset):
    def __init__(self, df, features):
        self.source, self.target = map_nodes(df)
        self.edge_features = features
        self.labels = df[LABEL_COL].astype(np.float32).values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.source[idx], dtype=torch.long),
            torch.tensor(self.target[idx], dtype=torch.long),
            torch.tensor(self.edge_features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

BATCH_SIZE = 2048

train_dataset = EdgeDataset(train_df, X_train)
val_dataset = EdgeDataset(val_df, X_val)
test_dataset = EdgeDataset(test_df, X_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

edge_feat_dim = X_train.shape[1]
print("edge_feat_dim:", edge_feat_dim)

edge_feat_dim: 28


In [11]:
# =========================
# MODEL
# =========================

class GraphEdgeClassifier(nn.Module):
    def __init__(self, num_nodes, edge_feat_dim, embedding_dim=64, hidden_dim=128, dropout=0.35):
        super().__init__()
        self.node_embedding = nn.Embedding(num_nodes, embedding_dim)
        input_dim = embedding_dim * 2 + edge_feat_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, source, target, edge_feat):
        src_emb = self.node_embedding(source)
        dst_emb = self.node_embedding(target)
        x = torch.cat([src_emb, dst_emb, edge_feat], dim=1)
        return self.mlp(x).squeeze(-1)

model = GraphEdgeClassifier(
    num_nodes=num_nodes,
    edge_feat_dim=edge_feat_dim,
    embedding_dim=64,
    hidden_dim=128,
    dropout=0.35,
).to(DEVICE)

model

GraphEdgeClassifier(
  (node_embedding): Embedding(126505, 64)
  (mlp): Sequential(
    (0): Linear(in_features=156, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.35, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.35, inplace=False)
    (8): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [12]:

# =========================
# LOSS AND OPTIMIZER: FOCAL LOSS
# =========================

class FocalLoss(nn.Module):
    """
    Binary Focal Loss for logits.

    alpha:
        Weight for positive/fraud class. Larger alpha increases fraud sensitivity.
    gamma:
        Focus factor. Larger gamma focuses more on hard examples.
    """
    def __init__(self, alpha=0.75, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        targets = targets.float()

        bce = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1.0 - probs) * (1.0 - targets)

        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        focal_factor = (1.0 - p_t).pow(self.gamma)

        loss = alpha_t * focal_factor * bce

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0

criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print("Loss function : FocalLoss")
print("FOCAL_ALPHA   :", FOCAL_ALPHA)
print("FOCAL_GAMMA   :", FOCAL_GAMMA)


Loss function : FocalLoss
FOCAL_ALPHA   : 0.75
FOCAL_GAMMA   : 2.0


In [13]:
# =========================
# TRAIN / EVAL HELPERS
# =========================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    n = 0
    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()
        logits = model(source, target, edge_feat)
        loss = criterion(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = label.size(0)
        total_loss += float(loss.item()) * bs
        n += bs
    return total_loss / max(n, 1)


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    all_probs = []
    all_labels = []
    for source, target, edge_feat, label in loader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)
        edge_feat = edge_feat.to(DEVICE)
        logits = model(source, target, edge_feat)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.append(probs)
        all_labels.append(label.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs)


def safe_auc(labels, probs, fn):
    try:
        return float(fn(labels, probs))
    except ValueError:
        return float("nan")


def compute_metrics(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, zero_division=0)),
        "recall": float(recall_score(labels, preds, zero_division=0)),
        "f1": float(f1_score(labels, preds, zero_division=0)),
        "roc_auc": safe_auc(labels, probs, roc_auc_score),
        "pr_auc": safe_auc(labels, probs, average_precision_score),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

In [14]:
# =========================
# TRAINING WITH VALIDATION MODEL SELECTION
# =========================

EPOCHS = 30
PATIENCE = 6
BEST_MODEL_PATH = OUTPUT_MODELS_DIR / "graph_focal_loss_upgrade_no_leakage_best.pt"

history = []
best_score = -1
best_epoch = 0
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

    val_labels, val_probs = predict_proba(model, val_loader)
    val_m = compute_metrics(val_labels, val_probs, threshold=0.5)

    # Model selection pakai PR-AUC karena fraud sangat imbalance.
    score = val_m["pr_auc"] if not np.isnan(val_m["pr_auc"]) else -1

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy@0.5": val_m["accuracy"],
        "val_precision@0.5": val_m["precision"],
        "val_recall@0.5": val_m["recall"],
        "val_f1@0.5": val_m["f1"],
        "val_roc_auc": val_m["roc_auc"],
        "val_pr_auc": val_m["pr_auc"],
    }
    history.append(row)
    print(row)

    if score > best_score:
        best_score = score
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
            break

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_history.csv", index=False)
history_df

{'epoch': 1, 'train_loss': 0.02137104669051722, 'val_accuracy@0.5': 0.8921176020767513, 'val_precision@0.5': 0.00167858253030774, 'val_recall@0.5': 0.5192307692307693, 'val_f1@0.5': 0.0033463469046291134, 'val_roc_auc': 0.7679210556063408, 'val_pr_auc': 0.0028145251632450325}
{'epoch': 2, 'train_loss': 0.012547012703300428, 'val_accuracy@0.5': 0.8603290872624582, 'val_precision@0.5': 0.0015359508495728137, 'val_recall@0.5': 0.6153846153846154, 'val_f1@0.5': 0.0030642535669826677, 'val_roc_auc': 0.7750507006732388, 'val_pr_auc': 0.0031503525576358784}
{'epoch': 3, 'train_loss': 0.010224935787058624, 'val_accuracy@0.5': 0.8034599105172425, 'val_precision@0.5': 0.001091703056768559, 'val_recall@0.5': 0.6153846153846154, 'val_f1@0.5': 0.002179539572265359, 'val_roc_auc': 0.7744159421808439, 'val_pr_auc': 0.0030240542410069613}
{'epoch': 4, 'train_loss': 0.008682146612245752, 'val_accuracy@0.5': 0.7696255005735214, 'val_precision@0.5': 0.000989522700814901, 'val_recall@0.5': 0.6538461538461

,epoch,train_loss,val_accuracy@0.5,val_precision@0.5,val_recall@0.5,val_f1@0.5,val_roc_auc,val_pr_auc
0,1,0.021371,0.892118,0.001679,0.519231,0.003346,0.767921,0.002815
1,2,0.012547,0.860329,0.001536,0.615385,0.003064,0.775051,0.003150
2,3,0.010225,0.803460,0.001092,0.615385,0.002180,0.774416,0.003024
3,4,0.008682,0.769626,0.000990,0.653846,0.001976,0.778717,0.003659
4,5,0.007876,0.749884,0.000938,0.673077,0.001874,0.776456,0.004217
5,6,0.007234,0.737388,0.000945,0.711538,0.001887,0.778940,0.005346
6,7,0.006740,0.744189,0.000970,0.711538,0.001937,0.779759,0.007697
7,8,0.006097,0.727118,0.000909,0.711538,0.001816,0.776783,0.008117
8,9,0.005838,0.747825,0.000931,0.673077,0.001859,0.776683,0.010361
9,10,0.005551,0.736764,0.000917,0.692308,0.001831,0.775758,0.011876


In [15]:
# =========================
# VALIDATION THRESHOLD SEARCH
# =========================

def build_threshold_table(labels, probs):
    quantile_thresholds = np.unique(np.quantile(probs, np.linspace(0, 1, 1001)))
    fixed_thresholds = np.array([0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5])
    thresholds = np.unique(np.concatenate([quantile_thresholds, fixed_thresholds]))

    rows = []
    for t in thresholds:
        m = compute_metrics(labels, probs, threshold=float(t))
        rows.append({"threshold": float(t), **m})
    return pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)


def select_threshold(df, strategy="fp_budget_then_recall", recall_target=0.6, max_fp_rate=0.05):
    max_fp = int(max_fp_rate * (df["tn"] + df["fp"]).max())

    if strategy == "best_f1":
        best = df.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
    elif strategy == "recall_target_min_fp":
        cand = df[df["recall"] >= recall_target].copy()
        if len(cand) == 0:
            best = df.sort_values(["recall", "f1"], ascending=False).iloc[0]
        else:
            best = cand.sort_values(["fp", "precision", "f1"], ascending=[True, False, False]).iloc[0]
    elif strategy == "fp_budget_then_recall":
        cand = df[df["fp"] <= max_fp].copy()
        if len(cand) == 0:
            best = df.sort_values(["fp", "recall", "f1"], ascending=[True, False, False]).iloc[0]
        else:
            # Dalam budget FP, pilih recall tertinggi, lalu precision/F1 tertinggi.
            best = cand.sort_values(["recall", "precision", "f1"], ascending=False).iloc[0]
    else:
        raise ValueError("Unknown threshold strategy")

    return float(best["threshold"]), best.to_dict(), max_fp

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
val_labels, val_probs = predict_proba(model, val_loader)
val_threshold_table = build_threshold_table(val_labels, val_probs)

THRESHOLD_STRATEGY = "fp_budget_then_recall"
RECALL_TARGET = 0.60
MAX_FP_RATE_ON_VAL_NORMAL = 0.05  # 5% normal validation boleh menjadi alert candidate.

BEST_THRESHOLD, best_threshold_row, val_fp_budget = select_threshold(
    val_threshold_table,
    strategy=THRESHOLD_STRATEGY,
    recall_target=RECALL_TARGET,
    max_fp_rate=MAX_FP_RATE_ON_VAL_NORMAL,
)

print("Best epoch:", best_epoch)
print("Best validation PR-AUC:", best_score)
print("Threshold strategy:", THRESHOLD_STRATEGY)
print("Validation FP budget:", val_fp_budget)
print("Best threshold:", BEST_THRESHOLD)
print(json.dumps(best_threshold_row, indent=2))

val_threshold_table.to_csv(OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_thresholds_validation.csv", index=False)
val_threshold_table.sort_values(["f1", "recall"], ascending=False).head(20)

Best epoch: 27
Best validation PR-AUC: 0.04284346666916614
Threshold strategy: fp_budget_then_recall
Validation FP budget: 7451
Best threshold: 0.7187257040739048
{
  "threshold": 0.7187257040739048,
  "accuracy": 0.9610206668947336,
  "precision": 0.004815133276010318,
  "recall": 0.5384615384615384,
  "f1": 0.00954491222089654,
  "roc_auc": 0.7402513793318815,
  "pr_auc": 0.04284346666916614,
  "tn": 143240.0,
  "fp": 5787.0,
  "fn": 24.0,
  "tp": 28.0
}


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
1009,0.953954,0.998833,0.093333,0.269231,0.138614,0.740251,0.042843,148891,136,38,14
1008,0.921424,0.997927,0.070234,0.403846,0.119658,0.740251,0.042843,148749,278,31,21
1007,0.900628,0.996955,0.051339,0.442308,0.092000,0.740251,0.042843,148602,425,29,23
1006,0.885765,0.995982,0.041876,0.480769,0.077042,0.740251,0.042843,148455,572,27,25
1005,0.873445,0.994983,0.033512,0.480769,0.062657,0.740251,0.042843,148306,721,27,25
1004,0.863046,0.993983,0.027933,0.480769,0.052798,0.740251,0.042843,148157,870,27,25
1003,0.853924,0.992984,0.023946,0.480769,0.045620,0.740251,0.042843,148008,1019,27,25
1002,0.846039,0.991984,0.020956,0.480769,0.040161,0.740251,0.042843,147859,1168,27,25
1001,0.839066,0.990985,0.018629,0.480769,0.035868,0.740251,0.042843,147710,1317,27,25
1000,0.832716,0.989985,0.016767,0.480769,0.032404,0.740251,0.042843,147561,1466,27,25


In [16]:

# =========================
# FINAL TEST EVALUATION
# =========================
# Test set hanya dipakai setelah model dan threshold selesai dipilih dari validation.

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
test_labels, test_probs = predict_proba(model, test_loader)

final_m = compute_metrics(
    test_labels,
    test_probs,
    threshold=BEST_THRESHOLD
)

final_result = {
    "model": "graph_focal_loss_upgrade_no_leakage",
    "loss": "focal_loss",
    "focal_alpha": FOCAL_ALPHA,
    "focal_gamma": FOCAL_GAMMA,
    "best_epoch": int(best_epoch),
    "best_validation_pr_auc": float(best_score),
    "threshold_strategy": THRESHOLD_STRATEGY,
    "best_threshold_from_validation": float(BEST_THRESHOLD),
    "threshold": float(BEST_THRESHOLD),
    **final_m,
}

with open(OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_final_result.json", "w") as f:
    json.dump(final_result, f, indent=2)

print(json.dumps(final_result, indent=2))


{
  "model": "graph_focal_loss_upgrade_no_leakage",
  "loss": "focal_loss",
  "focal_alpha": 0.75,
  "focal_gamma": 2.0,
  "best_epoch": 27,
  "best_validation_pr_auc": 0.04284346666916614,
  "threshold_strategy": "fp_budget_then_recall",
  "best_threshold_from_validation": 0.7187257040739048,
  "threshold": 0.7187257040739048,
  "accuracy": 0.5162027099543869,
  "precision": 0.00032575773328065763,
  "recall": 0.734375,
  "f1": 0.0006512265922143782,
  "roc_auc": 0.737198256811564,
  "pr_auc": 0.030904590051455797,
  "tn": 153864,
  "fp": 144232,
  "fn": 17,
  "tp": 47
}


In [17]:

# =========================
# TEST THRESHOLD TABLE - ANALYSIS ONLY
# =========================
# Test threshold table hanya untuk analisis laporan. Jangan memilih threshold dari sini.

test_threshold_rows = []
threshold_candidates = [
    0.001, 0.005, 0.01, 0.02, 0.05,
    0.1, 0.2, 0.3, 0.4, 0.5,
    BEST_THRESHOLD
]

for t in sorted(set([float(x) for x in threshold_candidates if pd.notna(x)])):
    test_threshold_rows.append({
        "threshold": float(t),
        **compute_metrics(test_labels, test_probs, threshold=float(t))
    })

test_threshold_df = pd.DataFrame(test_threshold_rows)
test_threshold_df.to_csv(
    OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_thresholds_test_analysis.csv",
    index=False
)
test_threshold_df


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.001000,0.023132,0.000216,0.984375,0.000432,0.737198,0.030905,6834,291262,1,63
1,0.005000,0.042598,0.000217,0.968750,0.000434,0.737198,0.030905,12639,285457,2,62
2,0.010000,0.054799,0.000216,0.953125,0.000433,0.737198,0.030905,16278,281818,3,61
3,0.020000,0.068779,0.000220,0.953125,0.000439,0.737198,0.030905,20446,277650,3,61
4,0.050000,0.093550,0.000222,0.937500,0.000444,0.737198,0.030905,27833,270263,4,60
5,0.100000,0.123202,0.000218,0.890625,0.000436,0.737198,0.030905,36677,261419,7,57
6,0.200000,0.170533,0.000222,0.859375,0.000445,0.737198,0.030905,50791,247305,9,55
7,0.300000,0.215747,0.000231,0.843750,0.000462,0.737198,0.030905,64273,233823,10,54
8,0.400000,0.265408,0.000237,0.812500,0.000475,0.737198,0.030905,79082,219014,12,52
9,0.500000,0.324369,0.000253,0.796875,0.000506,0.737198,0.030905,96663,201433,13,51


In [18]:
# =========================
# TEST THRESHOLD TABLE - ANALYSIS ONLY
# =========================
# Test threshold table hanya untuk analisis laporan. Jangan memilih threshold dari sini.

test_threshold_rows = []
for t in sorted(set([0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, BEST_THRESHOLD])):
    test_threshold_rows.append({"threshold": float(t), **compute_metrics(test_labels, test_probs, threshold=float(t))})

test_threshold_df = pd.DataFrame(test_threshold_rows)
test_threshold_df.to_csv(OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_thresholds_test_analysis.csv", index=False)
test_threshold_df

,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.001000,0.023132,0.000216,0.984375,0.000432,0.737198,0.030905,6834,291262,1,63
1,0.005000,0.042598,0.000217,0.968750,0.000434,0.737198,0.030905,12639,285457,2,62
2,0.010000,0.054799,0.000216,0.953125,0.000433,0.737198,0.030905,16278,281818,3,61
3,0.020000,0.068779,0.000220,0.953125,0.000439,0.737198,0.030905,20446,277650,3,61
4,0.050000,0.093550,0.000222,0.937500,0.000444,0.737198,0.030905,27833,270263,4,60
5,0.100000,0.123202,0.000218,0.890625,0.000436,0.737198,0.030905,36677,261419,7,57
6,0.200000,0.170533,0.000222,0.859375,0.000445,0.737198,0.030905,50791,247305,9,55
7,0.300000,0.215747,0.000231,0.843750,0.000462,0.737198,0.030905,64273,233823,10,54
8,0.400000,0.265408,0.000237,0.812500,0.000475,0.737198,0.030905,79082,219014,12,52
9,0.500000,0.324369,0.000253,0.796875,0.000506,0.737198,0.030905,96663,201433,13,51


In [19]:
# =========================
# PREDICTION RESULT TABLE
# =========================

test_results = pd.DataFrame({
    "true_label": test_labels.astype(int),
    "probability": test_probs,
    "prediction": (test_probs >= BEST_THRESHOLD).astype(int),
})

test_with_pred = pd.concat([test_df.reset_index(drop=True), test_results], axis=1)
candidate_fraud_full = test_with_pred[test_with_pred["prediction"] == 1].copy()

print("test_with_pred:", test_with_pred.shape)
print("candidate_fraud_full:", candidate_fraud_full.shape)
print("Total real fraud in test:", int(test_with_pred["true_label"].sum()))

test_with_pred.to_csv(OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_test_predictions.csv", index=False)
candidate_fraud_full.to_csv(OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_candidate_fraud.csv", index=False)

test_with_pred: (298160, 47)
candidate_fraud_full: (144279, 47)
Total real fraud in test: 64


In [20]:
# =========================
# TOP-K RANKING + LIFT VS RANDOM
# =========================
# Ini metric paling penting untuk research fraud candidate prioritization.

topk_results = []
total_real_fraud = int(test_with_pred["true_label"].sum())
total_test = len(test_with_pred)
fraud_rate = total_real_fraud / total_test if total_test > 0 else 0

k_values = [50, 100, 250, 500, 1000, 5000, 10000, 20000, 50000]
k_values = [k for k in k_values if k <= total_test]
ranked = test_with_pred.sort_values("probability", ascending=False).reset_index(drop=True)

for k in k_values:
    topk = ranked.head(k)
    tp = int((topk["true_label"] == 1).sum())
    fp = int((topk["true_label"] == 0).sum())
    fn = total_real_fraud - tp
    precision = tp / k if k > 0 else 0
    recall = tp / total_real_fraud if total_real_fraud > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    expected_random_tp = k * fraud_rate
    lift = tp / expected_random_tp if expected_random_tp > 0 else 0
    threshold_at_k = float(topk["probability"].min()) if len(topk) > 0 else np.nan

    topk_results.append({
        "top_k": k,
        "threshold_at_k": threshold_at_k,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "expected_random_tp": expected_random_tp,
        "lift_vs_random": lift,
    })

topk_df = pd.DataFrame(topk_results)
topk_df.to_csv(OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_topk_lift.csv", index=False)
topk_df

,top_k,threshold_at_k,tp,fp,fn,precision,recall,f1,expected_random_tp,lift_vs_random
0,50,0.999895,4,46,60,0.08000,0.062500,0.070175,0.010732,372.700000
1,100,0.999435,12,88,52,0.12000,0.187500,0.146341,0.021465,559.050000
2,250,0.998193,16,234,48,0.06400,0.250000,0.101911,0.053662,298.160000
3,500,0.996820,21,479,43,0.04200,0.328125,0.074468,0.107325,195.667500
4,1000,0.994776,24,976,40,0.02400,0.375000,0.045113,0.214650,111.810000
5,5000,0.985102,31,4969,33,0.00620,0.484375,0.012243,1.073249,28.884250
6,10000,0.975558,32,9968,32,0.00320,0.500000,0.006359,2.146499,14.908000
7,20000,0.958853,35,19965,29,0.00175,0.546875,0.003489,4.292997,8.152812
8,50000,0.912912,36,49964,28,0.00072,0.562500,0.001438,10.732493,3.354300


In [21]:
# =========================
# STAGE-2 RULE FILTERING ANALYSIS ONLY
# =========================
# Rule columns TIDAK dipakai sebagai fitur model.
# Di sini rule dipakai hanya setelah model membuat candidate, sebagai analisis stage-2 filtering.

filter_results = []
total_real_fraud = int(test_with_pred["true_label"].sum())

filter_sets = [("model_only", candidate_fraud_full)]

if "wash_score" in candidate_fraud_full.columns:
    filter_sets.extend([
        ("model + wash_score >= 1", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 1]),
        ("model + wash_score >= 2", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 2]),
        ("model + wash_score >= 3", candidate_fraud_full[candidate_fraud_full["wash_score"] >= 3]),
    ])

if "confidence_category" in candidate_fraud_full.columns:
    filter_sets.extend([
        ("model + confidence medium/high", candidate_fraud_full[candidate_fraud_full["confidence_category"].isin(["medium", "high"])]),
        ("model + confidence high", candidate_fraud_full[candidate_fraud_full["confidence_category"] == "high"]),
    ])

for rule_name, filtered_df in filter_sets:
    tp = int((filtered_df["true_label"] == 1).sum())
    fp = int((filtered_df["true_label"] == 0).sum())
    fn = total_real_fraud - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / total_real_fraud if total_real_fraud > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    filter_results.append({
        "filter": rule_name,
        "candidate_count": len(filtered_df),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

filter_df = pd.DataFrame(filter_results)
filter_df.to_csv(OUTPUT_METRICS_DIR / "graph_focal_loss_upgrade_no_leakage_stage2_filtering.csv", index=False)
filter_df

,filter,candidate_count,tp,fp,fn,precision,recall,f1
0,model_only,144279,47,144232,17,0.000326,0.734375,0.000651
1,model + wash_score >= 1,47,47,0,17,1.000000,0.734375,0.846847
2,model + wash_score >= 2,47,47,0,17,1.000000,0.734375,0.846847
3,model + wash_score >= 3,4,4,0,60,1.000000,0.062500,0.117647
4,model + confidence medium/high,0,0,0,64,0.000000,0.000000,0.000000
5,model + confidence high,0,0,0,64,0.000000,0.000000,0.000000


In [22]:

# =========================
# FINAL TEST RESULT
# =========================

print("=" * 60)
print("FINAL TEST RESULT")
print("=" * 60)

if "final_result" not in globals():
    raise RuntimeError("final_result belum dibuat. Jalankan cell FINAL TEST EVALUATION terlebih dahulu.")

display(pd.DataFrame([final_result]))


FINAL TEST RESULT


,model,loss,focal_alpha,focal_gamma,best_epoch,best_validation_pr_auc,threshold_strategy,best_threshold_from_validation,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,graph_focal_loss_upgrade_no_leakage,focal_loss,0.75,2.0,27,0.042843,fp_budget_then_recall,0.718726,0.718726,0.516203,0.000326,0.734375,0.000651,0.737198,0.030905,153864,144232,17,47


## Cara membaca hasil

- Jika threshold global menghasilkan FP besar, jangan langsung menyimpulkan model gagal total. Pada fraud rate sangat kecil, evaluasi utama harus melihat `PR-AUC`, `Top-K Recall`, dan `Lift vs Random`.
- `Stage-2 rule filtering` boleh dipakai sebagai filtering setelah model, tetapi jangan dipakai sebagai input feature karena itu menyebabkan label leakage.
- Jika Top-K memiliki lift tinggi, model layak diposisikan sebagai **fraud candidate ranking / prioritization system**, bukan final accusation system.

In [23]:

# =========================
# SUMMARY OUTPUT
# =========================

required_vars = ["final_result", "best_threshold_row", "BEST_THRESHOLD", "test_threshold_df", "topk_df", "filter_df", "history_df"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Variabel belum lengkap: {missing_vars}. Jalankan notebook dari atas memakai Kernel -> Restart & Run All.")

print("=" * 60)
print("FINAL TEST RESULT - FOCAL LOSS")
print("=" * 60)
display(pd.DataFrame([final_result]))

print("\n--- Best Validation Threshold ---")
display(pd.DataFrame([best_threshold_row]) if isinstance(best_threshold_row, dict) else best_threshold_row.to_frame().T)
print("Selected threshold:", BEST_THRESHOLD)

print("\n--- Test Threshold Table ---")
display(test_threshold_df)

print("\n--- Top-K Result ---")
display(topk_df)

print("\n--- Filtering Result ---")
display(filter_df)

print("\n--- Training History ---")
display(history_df.tail(10))


FINAL TEST RESULT - FOCAL LOSS


,model,loss,focal_alpha,focal_gamma,best_epoch,best_validation_pr_auc,threshold_strategy,best_threshold_from_validation,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,graph_focal_loss_upgrade_no_leakage,focal_loss,0.75,2.0,27,0.042843,fp_budget_then_recall,0.718726,0.718726,0.516203,0.000326,0.734375,0.000651,0.737198,0.030905,153864,144232,17,47



--- Best Validation Threshold ---


,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,0.718726,0.961021,0.004815,0.538462,0.009545,0.740251,0.042843,143240.0,5787.0,24.0,28.0


Selected threshold: 0.7187257040739048

--- Test Threshold Table ---


...

,top_k,threshold_at_k,tp,fp,fn,precision,recall,f1,expected_random_tp,lift_vs_random
0,50,0.999895,4,46,60,0.08000,0.062500,0.070175,0.010732,372.700000
1,100,0.999435,12,88,52,0.12000,0.187500,0.146341,0.021465,559.050000
2,250,0.998193,16,234,48,0.06400,0.250000,0.101911,0.053662,298.160000
3,500,0.996820,21,479,43,0.04200,0.328125,0.074468,0.107325,195.667500
4,1000,0.994776,24,976,40,0.02400,0.375000,0.045113,0.214650,111.810000
5,5000,0.985102,31,4969,33,0.00620,0.484375,0.012243,1.073249,28.884250
6,10000,0.975558,32,9968,32,0.00320,0.500000,0.006359,2.146499,14.908000
7,20000,0.958853,35,19965,29,0.00175,0.546875,0.003489,4.292997,8.152812
8,50000,0.912912,36,49964,28,0.00072,0.562500,0.001438,10.732493,3.354300



--- Filtering Result ---


,filter,candidate_count,tp,fp,fn,precision,recall,f1
0,model_only,144279,47,144232,17,0.000326,0.734375,0.000651
1,model + wash_score >= 1,47,47,0,17,1.000000,0.734375,0.846847
2,model + wash_score >= 2,47,47,0,17,1.000000,0.734375,0.846847
3,model + wash_score >= 3,4,4,0,60,1.000000,0.062500,0.117647
4,model + confidence medium/high,0,0,0,64,0.000000,0.000000,0.000000
5,model + confidence high,0,0,0,64,0.000000,0.000000,0.000000



--- Training History ---


,epoch,train_loss,val_accuracy@0.5,val_precision@0.5,val_recall@0.5,val_f1@0.5,val_roc_auc,val_pr_auc
20,21,0.002187,0.820806,0.001197,0.615385,0.002390,0.758919,0.036584
21,22,0.002145,0.819418,0.001151,0.596154,0.002298,0.762773,0.033807
22,23,0.002105,0.836463,0.001230,0.576923,0.002455,0.760101,0.031256
23,24,0.001763,0.826032,0.001156,0.576923,0.002308,0.762750,0.030522
24,25,0.001723,0.878038,0.001649,0.576923,0.003289,0.752881,0.040826
25,26,0.001506,0.842439,0.001319,0.596154,0.002633,0.753180,0.038740
26,27,0.001533,0.846437,0.001354,0.596154,0.002701,0.740251,0.042843
27,28,0.001390,0.852642,0.001411,0.596154,0.002814,0.743422,0.042494
28,29,0.001344,0.851723,0.001402,0.596154,0.002797,0.744711,0.042468
29,30,0.001179,0.844525,0.001337,0.596154,0.002668,0.743807,0.040606
